In [ ]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from tqdm import tqdm
import colorir as cl
from analyses.rust import contact
from analyses import io
from pathlib import Path

## Static clusters (ji dataset)

In [ ]:
latpaths = []
for energy in Path("../runs/ji/").iterdir():
    for act in energy.iterdir():
        latpaths.append(act / "lattices")
latpaths

In [ ]:
latdf = pl.read_parquet(latpaths, include_file_paths="path")
latdf

In [ ]:
neigh_maps = {}
for (path,), clat in tqdm(latdf.group_by("path"), total=latdf["path"].n_unique()):
    neigh_maps[path] = contact.neighbour_map(clat.select(pl.exclude("path")))
neigh_maps

In [ ]:
pathdf = pl.DataFrame(list(neigh_maps.keys())).with_columns(list=pl.col("column_0").str.split("/")).with_columns(
    wtime=pl.col("list").list.last().str.replace(".parquet", "").cast(int),
    act=pl.col("list").list.get(-3).str.replace("act-", "").cast(int),
    gamma=20 - pl.col("list").list.get(-4).str.replace("energy-", "").cast(int),
).sort(["gamma", "act", "wtime"])
pathdf

In [ ]:
def jaccards_index(set1: set, set2):
    union = len(set1.union(set2))
    if union == 0:
        return None
    inter = len(set1.intersection(set2))
    return inter / union

In [ ]:
jis = []
rows = list(pathdf.iter_rows())
for row1, row2 in tqdm(zip(rows, rows[1:]), total=len(rows) - 1):
    if row1[-1] != row2[-1] or row1[-2] != row2[-2]:
        continue

    index1 = row1[2:]
    index2 = row2[2:]
    newindex = (index1[0], index2[0], *index1[1:])
    
    neighmap1 = neigh_maps[row1[0]]
    neighmap2 = neigh_maps[row2[0]]

    for spin, neighs1 in neighmap1.items():
        discard = {"s", "m"}  # We don't necessarily need to discard these but its prob a good idea
        ji = jaccards_index(set(neighs1) - discard, set(neighmap2[spin]) - discard)
        jis.append((*newindex, spin, ji))
jidf = pl.from_records(jis, orient="row", schema=["time1", "time2", "act", "gamma", "spin", "ji"])
jidf = jidf.with_columns(jd=1 - pl.col("ji"))
jidf

In [ ]:
aggdf = (
    jidf
        .filter(~pl.col("spin").is_in(["s", "m"]))
        .group_by(["gamma", "act"])
        .agg(
            mean=pl.col("jd").mean(),
            std=pl.col("jd").std(),
            med=pl.col("jd").median(),
            min=pl.col("jd").min(),
            max=pl.col("jd").max(),
        )
).sort(["gamma", "act"])
aggdf

In [ ]:
actdf = aggdf.filter(act=160)
fig = go.Figure()
fig.add_traces([
    go.Scatter(
        x=pl.concat([actdf["gamma"], actdf["gamma"][::-1]]),
        y=pl.concat([actdf["mean"] + actdf["std"], (actdf["mean"] - actdf["std"])[::-1]]),
        fill="toself",
        fillcolor="rgba(0, 0, 0, 0.2)",
        line_color="rgba(0, 0, 0, 0)"
    ),
    go.Scatter(
        x=actdf["gamma"],
        y=actdf["mean"],
        line_color="#000000"
    ),
])
fig.update_layout(
    template="plotly_white",
    width=500,
    height=500
    #showlegend=False
)

## Moving cluster (ji pol dataset)

In [ ]:
celldf = pl.read_parquet("../runs/ji_pol/**/cells/*.parquet", include_file_paths="path")
celldf = celldf.with_columns(list=pl.col("path").str.split("/")).select(
    pl.exclude(["path", "list", "neighbors"]),
    pl.col("neighbors").str.split(" ").list.set_difference(["m", "s"]),
    wtime=pl.col("list").list.last().str.replace(".parquet", "").cast(int),
    replica=pl.col("list").list.get(-3).cast(int),
    gamma=20 - pl.col("list").list.get(-4).str.replace("energy-", "").cast(int)
).sort(["gamma", "replica", "time"])
celldf

In [ ]:
celldf.filter(pl.col("tot_kact").is_infinite()).sort("gamma", "replica", "time", "index")

In [ ]:
shift = 1
setdf = celldf.group_by(["gamma", "replica", "index"]).agg(
    union_size=pl.col("neighbors").list.set_union(pl.col("neighbors").shift(-shift)).list.len(),
    inter_size=pl.col("neighbors").list.set_intersection(pl.col("neighbors").shift(-shift)).list.len(),
)
size = setdf["union_size"].list.len().unique().item()
setdf = setdf.with_columns(
    union_size=pl.col("union_size").list.to_array(size),
    inter_size=pl.col("inter_size").list.to_array(size)
)
setdf

In [ ]:
jidf = setdf.select(
    pl.exclude("^.*size$"),
    ji=(pl.col("inter_size") / pl.col("union_size")),
    idx=pl.int_ranges(0, pl.col("inter_size").arr.len())
).explode(pl.col(["ji", "idx"]))
jidf

In [ ]:
indexers = ["time", "gamma", "replica", "index"]
chemjidf = (
    celldf
        .select(pl.col("time").sort().unique())
        .with_row_index(name="idx")
        .join(jidf, on="idx")
        .select(pl.exclude("idx"))
        .join(celldf.select(*indexers, avg_chem=pl.col("chem_mass") / pl.col("area")), on=indexers)
        .with_columns(jd=1 - pl.col("ji"))
)
assert(chemjidf.filter(time=pl.col("time").max())["ji"].unique().item() is None)
chemjidf

In [ ]:
aggdf = (
    chemjidf
        .drop_nans()
        .group_by(["gamma"])
        .agg(
            mean=pl.col("jd").mean(),
            std=pl.col("jd").std(),
            med=pl.col("jd").median(),
            min=pl.col("jd").min(),
            max=pl.col("jd").max(),
        )
).sort(["gamma"])
aggdf

In [ ]:
fig = go.Figure()
fig.add_traces([
    go.Scatter(
        x=pl.concat([aggdf["gamma"], aggdf["gamma"][::-1]]),
        y=pl.concat([aggdf["mean"] + aggdf["std"], (aggdf["mean"] - aggdf["std"])[::-1]]),
        fill="toself",
        fillcolor="rgba(0, 0, 0, 0.2)",
        line_color="rgba(0, 0, 0, 0)"
    ),
    go.Scatter(
        x=aggdf["gamma"],
        y=aggdf["mean"],
        line_color="#000000"
    ),
])
fig.update_layout(
    template="plotly_white",
    width=500,
    height=500,
    xaxis_title="gamma",
    yaxis_title="Jaccard distance",
    showlegend=False
)
fig.write_image("../plots/jd_gamma.svg")
fig

In [ ]:
bins = np.linspace(0, chemjidf["avg_chem"].max(), 25)
chemaggdf = (
    chemjidf
        .drop_nans()
        .with_columns(
            chem_bin=pl.col("avg_chem").cut(bins[:-1], labels=bins.astype(str)).cast(str).cast(float)
        )
        .group_by([
            "gamma", 
            "chem_bin"
        ])
        .agg(pl.col("jd").mean())
        .sort(["gamma", "chem_bin"])
)
chemaggdf

In [ ]:
colors = cl.StackPalette.load("carnival").resize(11)
fig = px.line(
    chemaggdf, 
    x="chem_bin",
    y="jd",
    color="gamma",
    color_discrete_sequence=colors
).update_layout(
    template="plotly_white",
    width=500,
    height=500,
    xaxis_title="chem. concentration",
    yaxis_title="mean Jaccard distance"
)
fig.write_image("../plots/jd_chem.svg")
fig

In [ ]:
aggdf = celldf.group_by("gamma").agg(
    act_mean=pl.col("tot_act").mean(),
    k_act_mean=pl.col("tot_kact").mean(),
    std=pl.col("tot_act").std(),
).sort("gamma")
aggdf

In [ ]:
fig = px.bar(aggdf, x="gamma", y="act_mean", color="gamma", color_continuous_scale=cl.Grad(colors).to_plotly_colorscale())
fig.update_layout(
    template="plotly_white",
    yaxis_title="mean act of cells",
    width=400,
    height=300
)
io.save_plot(fig, "../plots/act_gamma")

In [ ]:
fig = px.bar(aggdf, x="gamma", y="k_act_mean", color="gamma", color_continuous_scale=cl.Grad(colors).to_plotly_colorscale())
fig.update_layout(
    template="plotly_white",
    yaxis_title="mean kernel act of cells",
    width=400,
    height=300
)
io.save_plot(fig, "../plots/kernel-act_gamma")

In [ ]:
from io import StringIO

def minmax_scaler(col):
    return (col - col.min()) / (col.max() - col.min())

def rel_scaler(col):
    return col / col.mean()

max_chem = (2 * 500 ** 2) ** 0.5 - 180
rel_bins = np.linspace(0, 1, 15)
indexers = ["gamma", "replica", "time", "index"]
cellmap = celldf.with_columns(
    cell_chem=pl.col("chem_mass") / pl.col("area"),
).with_columns(
    cluster_chem=pl.col("cell_chem").mean().over("gamma", "replica", "time"),
).with_columns(
    ptime=pl.col("time").filter(
        pl.col("cluster_chem") >= max_chem
    ).min().over("gamma", "replica")
).filter(
    # Filter close to peak
    500 > pl.col("cluster_chem"),
    # Filter close to start
    200 < pl.col("cluster_chem"),
    # Filter time to exclude reforming clusters
    pl.col("time") < pl.col("ptime"),
    # Filter loners
    0 < pl.col("neighbors").list.len() - pl.col("med_neighbor") - pl.col("solid_neighbor"),
    # Filter bad values of tot_kact (we need to rerun the sims to fix these)
    pl.col("tot_kact").is_finite(),
    # Filter cells at lattice border
    solid_neighbor=False
).with_columns(
    rel_act=rel_scaler(pl.col("tot_kact")).over(["gamma", "replica", "time"]),
    rel_chem=minmax_scaler(pl.col("cell_chem")).over(["gamma", "replica", "time"]),
).with_columns(
    chem_bin=pl.col("rel_chem").cut(rel_bins[:-1], labels=rel_bins.round(2).astype(str))
).sort(indexers)
cellmap

In [ ]:
aggdf = cellmap.filter(med_neighbor=True).group_by(["gamma", "chem_bin"]).mean().sort(["gamma", "chem_bin"])
aggdf

In [ ]:
# This is U-shaped because there are more cells touching the medium at rel_chem 1 and 0
# Or maybe not, filtering only border cells doesnt get rid of it...
fig = px.line(
    aggdf, 
    x="chem_bin", 
    y="rel_act", 
    color="gamma",
    color_discrete_sequence=colors
).update_layout(
    width=600,
    height=400,
    template="plotly_white",
    xaxis_dtick=1,
    xaxis_title="relative chem. concentration",
    yaxis_title="relative effective act"
)
io.save_plot(fig, "../plots/rel-chem_rel-act")

In [ ]:
cellmap.filter(time=pl.col("wtime"), chem_bin="0.0", gamma=10).sort("rel_act").tail(5).select(indexers + ["rel_act"])

In [ ]:
aggs = ["gamma", "replica", "index", "time"]
meandf = cellmap.with_columns(
    mean_rel_act=pl.col("rel_act").mean().over(aggs),
    mean_rel_chem=pl.col("rel_chem").mean().over(aggs),
    pleader=((pl.col("rel_act") > 1)).mean().over(aggs)
).filter(
    # Filter after obtaining mean
    med_neighbor=True
).group_by(aggs, maintain_order=True).mean()
meandf

In [ ]:
px.histogram(celldf["tot_kact"]).show(renderer="png")

In [ ]:
# No leaders and followers...
# We should try "mean time with act above cluster mean"
fig = px.histogram(meandf, x="mean_rel_chem", facet_col="gamma", histnorm="probability", nbins=25)
io.save_plot(fig, "../plots/rel_act_distr")

In [ ]:
fig = px.histogram(meandf, x="pleader", facet_col="gamma", histnorm="probability", nbins=25)
fig

In [ ]:
px.line(
    cellmap.filter(
        pl.col("index") < 5, 
        replica=0
    ), 
    x="time",
    y="rel_act",
    color="index", 
    facet_row="gamma",
    height=1000
)

In [ ]:
px.line(
    cellmap.filter(
        pl.col("index") < 5, 
        replica=0
    ), 
    x="time",
    y="rel_chem",
    color="index", 
    facet_row="gamma",
    height=1000
)

In [ ]:
def plot_indexes(gamma, replica, time):
    clat = pl.read_parquet(f"../runs/ji_pol/energy-{20 - gamma}/{replica}/lattices/{time:0>10}.parquet")
    return px.imshow(clat[::-1], width=500, height=500)

plot_indexes(10, 0, 2800000)

In [ ]:
match = "300000"  # Use 50000 for better res
actdf = None
for energy in tqdm(range(0, 21, 2)):
    clats = pl.read_parquet(fr"../runs/ji_pol/energy-{energy}/**/lattices/*{match}.parquet", include_file_paths="path").sort("path")
    alats = pl.read_parquet(fr"../runs/ji_pol/energy-{energy}/**/act_lattices/*{match}.parquet", include_file_paths="path").sort("path")
    chlats = pl.read_parquet(fr"../runs/ji_pol/energy-{energy}/**/chem_lattices/*{match}.parquet", include_file_paths="path").sort("path")
    c_no_path = clats.select(pl.exclude("path"))
    a_no_path = alats.select(pl.exclude("path"))
    c_array = c_no_path.to_numpy().astype("<U3")
    
    mask = ~np.isin(c_array, ["s", "m"])
    cells = c_array[mask].astype(np.uint32)
    acts = a_no_path.to_numpy()[mask]
    # Always need to reshape with F order to restore the matrix
    kerneled_acts = np.reshape(np.array(contact.kernel_act(c_no_path, a_no_path, 1, True)[1]), mask.shape, order="F")[mask]
    chems = chlats.select(pl.exclude("path")).to_numpy()[mask]

    row_mask = np.sum(mask, axis=1)
    pos = np.where(mask)
    plists = pl.Series(np.repeat(clats["path"], row_mask)).str.split("/")
    times = plists.list.get(-1).str.replace(".parquet", "").cast(pl.UInt32)
    replicas = plists.list.get(-3).cast(pl.UInt32)
    gamma = 20 - energy
    
    edf = pl.DataFrame({
        "index": cells,
        "time": times,
        "replica": replicas,
        "gamma": gamma,
        "x": pos[0] % 500,
        "y": pos[1],
        "act": acts,
        "k_act": kerneled_acts,
        "chem": chems, 
    })
    if actdf is None:
        actdf = edf
    else:
        actdf = pl.concat([actdf, edf])
actdf

In [ ]:
bins = np.linspace(0, actdf["chem"].max(), 25)
aggdf = actdf.filter(
    pl.col("time") > 1e4
).with_columns(
    chem_bin=pl.col("chem").cut(bins[:-1], labels=bins.astype(str)).cast(str).cast(float)
).group_by(["gamma", "chem_bin"]).agg(
    act_mean=pl.col("act").mean(),
    k_act_mean=pl.col("k_act").mean(),
    act_std=pl.col("act").std(),
).sort(["gamma", "chem_bin"])
aggdf

In [ ]:
# Mean act stays sable throughout the field, what changes is how it is polarized
colors = cl.StackPalette.load("carnival").resize(11)
fig = px.line(
    aggdf,
    x="chem_bin", 
    y="act_mean", 
    color="gamma", 
    color_discrete_sequence=colors
)
fig.update_layout(
    template="plotly_white",
    width=600,
    height=400,
    xaxis_title="chem. concentration",
    yaxis_title="mean act of cells"
)
io.save_plot(fig, "../plots/act_chem")

In [ ]:
fig = px.line(
    aggdf, 
    x="chem_bin",
    y="k_act_mean",
    color="gamma",
    color_discrete_sequence=colors
)
fig.update_layout(
    template="plotly_white",
    width=600,
    height=400,
    xaxis_title="chem. concentration",
    yaxis_title="mean g(act) of cells"
)
io.save_plot(fig, "../plots/kernel-act_chem")

In [ ]:
borderdf = actdf.join(
    cellmap.filter(
        gamma=10,
        replica=0,
        med_contact=True
    ).select(indexers + ["rel_act", "rel_chem"]),
    on=indexers
)
borderdf

In [ ]:
indexes = (borderdf["time"].to_numpy() // 25000, borderdf["x"].to_numpy(), borderdf["y"].to_numpy())
timearray = np.full((1 + (borderdf["time"].max() // 25000), 500, 500), np.nan)
timearray[indexes] = borderdf["rel_act"]
timearray = timearray[~np.all(np.isnan(timearray), axis=(1, 2))]
timearray

In [ ]:
px.imshow(
    timearray[:, ::-1, :],
    animation_frame=0,
    height=500
)